# 解答④ 評価指標

> **講師用**: 演習 `ex_04_evaluation.ipynb` の完全解答です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rouge_score import rouge_scorer
from bert_score import score as bert_score

plt.rcParams['font.family'] = 'IPAexGothic'
plt.rcParams['axes.unicode_minus'] = False

reference = '機械学習はデータからパターンを学習し、新しいデータに対して予測を行う人工知能の技術です。教師あり・教師なし・強化学習の3種類があります。'

hypotheses = {
    '完全一致':  '機械学習はデータからパターンを学習し、新しいデータに対して予測を行う人工知能の技術です。教師あり・教師なし・強化学習の3種類があります。',
    '言い換え':  '機械学習とは、大量のデータからルールを自動的に発見してAIに予測させる手法です。学習の種類には監視あり、監視なし、報酬学習があります。',
    '不完全':    '機械学習はデータを使う技術です。',
}

In [ ]:
# 解答: ROUGE スコア
scorer_rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

rouge_scores = {}
for name, hyp in hypotheses.items():
    scores = scorer_rouge.score(reference, hyp)
    rouge_scores[name] = scores['rougeL'].fmeasure

print('=== ROUGE-L スコア ===')
for name, score in rouge_scores.items():
    print(f'{name:8s}: {score:.3f}')

In [ ]:
# 解答: BERTScore
hyp_list = list(hypotheses.values())
ref_list  = [reference] * len(hyp_list)

P, R, F1 = bert_score(
    hyp_list, ref_list,
    lang='ja',
    model_type='bert-base-multilingual-cased',
    verbose=False,
)

bert_scores = {}
for i, name in enumerate(hypotheses.keys()):
    bert_scores[name] = F1[i].item()

print('=== BERTScore F1 ===')
for name, score in bert_scores.items():
    print(f'{name:8s}: {score:.3f}')

In [ ]:
# 解答: 可視化
names  = list(hypotheses.keys())
r_vals = [rouge_scores[n]  for n in names]
b_vals = [bert_scores[n]   for n in names]

x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, r_vals, w, label='ROUGE-L', color='#1A6B52')
ax.bar(x + w/2, b_vals, w, label='BERTScore F1', color='#0D4A38')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0, 1.05)
ax.set_ylabel('スコア')
ax.set_title('ROUGE-L vs BERTScore F1 の比較')
ax.legend()
for i, (r, b) in enumerate(zip(r_vals, b_vals)):
    ax.text(i - w/2, r + 0.01, f'{r:.2f}', ha='center', fontsize=9)
    ax.text(i + w/2, b + 0.01, f'{b:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\n考察:')
print('「言い換え」: ROUGE が低くても BERTScore は高い → 意味的に正しい言い換えを評価できている')
print('「不完全」  : ROUGE も BERTScore も低い → 情報量が少ない回答を両指標とも正しく評価できている')